# Using `mainfile.xlsx` for Alloy Extension Fitting

This notebook demonstrates how to drive the alloy extension fitting workflow using
`mainfile.xlsx` instead of hardcoded Python dictionaries.

## What is alloy mode?

In alloy mode the energetic surface-energy parameters (**A0**, **B0**, **l0**) for
alloy compositions are *blended* from the fitted pure-element values via rule of
mixtures.  Only **K0** varies per dataset; all other parameters are material-level.

| Parameter type | Per | Parameters |
|---|---|---|
| Process | Dataset | K0 |
| Material | Material | alpha1, L0, GrainSize_200, Sigma0, BetaD, Mfda, Di, A0, B0, l0 |

**Important:** Pure elements must appear before alloy compositions in the mainfile rows.

## mainfile.xlsx row layout (same as general mode)

| Row | Content |
|-----|--------|
| 1 | Column numbers |
| 2 | Parameter names |
| 3 | Variability type: `0` = shared per material, `1` = per dataset |
| 4 | Bound type: `3` = additive, `4` = multiplicative, `5` = zero-lower |
| 5 | Bound magnitude |
| 6+ | Material rows (pure elements first, then alloys) |

**Materials covered here:** Cr, W (pure) + Cr-25W, Cr-50W (alloys) — Su data

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

import torch
import torch.nn as nn

%matplotlib inline

# Add repo root to path
REPO_ROOT = Path('..').resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from kmorfs import (
    load_from_database,
    load_from_mainfile_data,
    AlloyMaterialDependentExtension,
    AlloySTFModel,
    read_mainfile,
    parse_mainfile_alloy,
    compute_bounds,
)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')

## Step 1 — Inspect the raw mainfile

In [ ]:
SCRIPT_DIR      = Path('..').resolve()          # alloy_extension_stress_thickness/
MAINFILE_PATH   = SCRIPT_DIR / 'mainfile.xlsx'
MAINFILE_DATA_DIR = SCRIPT_DIR / 'mainfile_data'
SOURCE_CSV      = REPO_ROOT / 'data' / 'source.csv'
EXPERIMENTS_CSV = REPO_ROOT / 'data' / 'all_experiments.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    matplotlib.rcParams['font.family'] = 'Times New Roman'
except Exception:
    pass

COLORS = np.array([
    '#5B9BD5', '#A5D6A7', '#F1C40F', '#E74C3C',
    '#9B59B6', '#F39C12', '#1F77B4', '#BDC3C7'
])

print(f'Mainfile path  : {MAINFILE_PATH}')
print(f'Mainfile exists: {MAINFILE_PATH.exists()}')
print(f'Device         : {DEVICE}')

In [ ]:
# Display raw mainfile structure
raw_df = read_mainfile(MAINFILE_PATH)
print('Raw mainfile (first 12 rows):')
display(raw_df.head(12))

print('\nRow legend:')
print('  Row 0 : column numbers')
print('  Row 1 : parameter names')
print('  Row 2 : variability type (0=material, 1=dataset)')
print('  Row 3 : bound type (3=additive, 4=multiplicative, 5=zero-lower)')
print('  Row 4 : bound magnitude')
print('  Row 5+: material rows — pure elements FIRST, then alloys')

## Step 2 — Parse with `parse_mainfile_alloy`

`parse_mainfile_alloy()` has the same signature as `parse_mainfile_general()` — the
difference is the column set expected (no `SigmaC` / `Ea`; K0 is a per-dataset
parameter, and grain/alpha columns are material-level).

In [ ]:
mainfile_params = parse_mainfile_alloy(MAINFILE_PATH)

print('Keys returned by parse_mainfile_alloy():')
for k, v in mainfile_params.items():
    if isinstance(v, dict):
        print(f'  {k}: dict with {len(v)} entries -> {list(v.keys())}')
    elif isinstance(v, list):
        print(f'  {k}: list of length {len(v)}')
    else:
        print(f'  {k}: {v}')

# The material order from mainfile is critical: pure elements must come first
print('\nMaterial order in mainfile (pure elements first, then alloys):')
materials_in_mainfile = list(mainfile_params['material_defaults'].keys())
for i, m in enumerate(materials_in_mainfile):
    is_pure = '-' not in m
    kind = 'pure element' if is_pure else 'alloy'
    print(f'  [{i}] {m}  ({kind})')

In [ ]:
# Inspect parameter defaults from mainfile
print('Material-level defaults (vtype=0):')
mat_df = pd.DataFrame(mainfile_params['material_defaults']).T
display(mat_df)

print('\nProcess defaults (vtype=1, per-dataset K0):')
proc_df = pd.DataFrame(mainfile_params['process_defaults']).T
display(proc_df)

print('\nBound configuration:')
bound_df = pd.DataFrame({
    'bound_type': mainfile_params['bound_types'],
    'bound_mag' : mainfile_params['bound_mags'],
})
bound_df['meaning'] = bound_df['bound_type'].map({
    3: 'additive ±mag',
    4: 'multiplicative ×(1±mag)',
    5: 'zero-lower [0, val×(1+mag)]',
})
display(bound_df)

## Step 3 — Detect pure element count and load data

The number of pure elements is auto-detected from mainfile rows (any material
without a `-` is treated as pure).  Mainfile ordering is validated so that
pure elements always precede alloys.

In [ ]:
def _is_pure_element(name):
    return '-' not in name

materials = list(mainfile_params['material_defaults'].keys())
n_pure_elements = sum(1 for m in materials if _is_pure_element(m))

# Validate ordering: pure elements must all come before any alloy
for i, m in enumerate(materials):
    if _is_pure_element(m) and i >= n_pure_elements:
        raise ValueError(
            f"Pure element '{m}' at index {i} appears after alloys. "
            "Reorder mainfile.xlsx so pure elements come first."
        )

print(f'Materials        : {materials}')
print(f'Pure elements    : {materials[:n_pure_elements]}')
print(f'Alloys           : {materials[n_pure_elements:]}')
print(f'n_pure_elements  : {n_pure_elements}')

In [ ]:
dataset_process_defaults = None

if 'filenames' in mainfile_params:
    # ── File-based mode ──────────────────────────────────────────────────────
    print(f'[File-based mode] Loading {len(mainfile_params["filenames"])} datasets'
          f' from {MAINFILE_DATA_DIR}')
    fit_data, process_condition, experiment_labels = load_from_mainfile_data(
        data_dir=MAINFILE_DATA_DIR,
        filenames=mainfile_params['filenames'],
        material_names=mainfile_params['material_names'],
    )
    dataset_process_defaults = mainfile_params['dataset_process_defaults']

else:
    # ── Database mode ────────────────────────────────────────────────────────
    print(f'[Database mode] Querying database for: {materials}')
    fit_data, process_condition, experiment_labels = load_from_database(
        source_path=SOURCE_CSV,
        experiments_path=EXPERIMENTS_CSV,
        materials=materials,
        data_sources=None,   # None = all available sources
    )

print(f'\nFound {len(experiment_labels)} experiments:')
for label in experiment_labels:
    print(f'  {label}')
print(f'\nTotal data points: {len(fit_data)}')
print('\nProcess conditions (first 6 rows):')
display(process_condition.head(6))

## Step 4 — Build config, alloy extension, and parameters

In [ ]:
_FALLBACK_GENERIC = {
    'K0': 0, 'alpha1': 0.02, 'L0': 10, 'GrainSize_200': 15,
    'Sigma0': 10, 'BetaD': 0.05, 'Mfda': 20, 'Di': 0.3,
    'A0': -3, 'B0': -8, 'l0': 0.8,
}

def get_material_params(material, mainfile_params):
    mat_defaults = mainfile_params.get('material_defaults', {})
    proc_defaults = mainfile_params.get('process_defaults', {})
    if material in mat_defaults or material in proc_defaults:
        params = _FALLBACK_GENERIC.copy()
        params.update(mat_defaults.get(material, {}))
        params.update(proc_defaults.get(material, {}))
        return params
    return _FALLBACK_GENERIC.copy()


def build_config(process_condition, experiment_labels, mainfile_params=None,
                 dataset_process_defaults=None):
    material_names = [label.split('_')[0] for label in experiment_labels]
    process_cols  = ['K0']
    material_cols = ['alpha1', 'L0', 'GrainSize_200', 'Sigma0',
                     'BetaD', 'Mfda', 'Di', 'A0', 'B0', 'l0']
    rows = []
    seen_materials = set()
    for i, label in enumerate(experiment_labels):
        mat = material_names[i]
        params = get_material_params(mat, mainfile_params) if mainfile_params else _FALLBACK_GENERIC.copy()
        mt = process_condition.iloc[i]['Melting_T']
        row = {
            'Fit_data': label,
            'R': process_condition.iloc[i]['R'],
            'T': process_condition.iloc[i]['T'],
            'P': process_condition.iloc[i]['P'],
            'Melting_T': mt,
            'SigmaC': 0,  # fixed in alloy mode
        }
        if dataset_process_defaults is not None:
            ds_proc = dataset_process_defaults[i]
            for col in process_cols:
                row[col] = ds_proc.get(col, params[col])
        else:
            for col in process_cols:
                row[col] = params[col]
        if mt not in seen_materials:
            for col in material_cols:
                row[col] = params[col]
            seen_materials.add(mt)
        else:
            for col in material_cols:
                row[col] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def setup_parameters(mainfile, mainfile_params=None, n_pure_elements=None):
    process_condition = mainfile[['R', 'T', 'P', 'Melting_T']]
    material_cols = ['alpha1', 'L0', 'GrainSize_200', 'Sigma0',
                     'BetaD', 'Mfda', 'Di', 'A0', 'B0', 'l0']
    initial_process   = mainfile[['K0']]
    initial_materials = mainfile[material_cols].dropna()
    x_vector = np.concatenate([initial_materials.values.flatten(),
                                initial_process.values.flatten()])

    # Default bound multipliers
    # [alpha1, L0, GrainSize_200, Sigma0, BetaD, Mfda, Di, A0, B0, l0]
    materials_bound = np.array([3, 2, 0.5, 3, 2, 0.5, 2, 2, 0.5, 0.2])
    process_bound   = np.array([300])  # K0 ±300

    if mainfile_params:
        bound_types = mainfile_params.get('bound_types', {})
        bound_mags  = mainfile_params.get('bound_mags',  {})

        materials_lb = initial_materials.copy().astype(float)
        materials_ub = initial_materials.copy().astype(float)
        for i, col in enumerate(material_cols):
            default_bt = 5 if i in [0, 1, 4, 6] else 4
            bt = bound_types.get(col, default_bt)
            bm = bound_mags.get(col, materials_bound[i])
            lb, ub = compute_bounds(bt, bm, initial_materials.iloc[:, i].values)
            materials_lb.iloc[:, i] = lb
            materials_ub.iloc[:, i] = ub

        process_lb = initial_process.copy().astype(float)
        process_ub = initial_process.copy().astype(float)
        bt = bound_types.get('K0', 3)
        bm = bound_mags.get('K0', process_bound[0])
        lb, ub = compute_bounds(bt, bm, initial_process.iloc[:, 0].values)
        process_lb.iloc[:, 0] = lb
        process_ub.iloc[:, 0] = ub
    else:
        materials_lb = initial_materials.copy().astype(float)
        materials_ub = initial_materials.copy().astype(float)
        for i in [0, 1, 4, 6]:
            materials_lb.iloc[:, i] = np.minimum(0, initial_materials.iloc[:, i] * (1 + materials_bound[i]))
            materials_ub.iloc[:, i] = np.maximum(0, initial_materials.iloc[:, i] * (1 + materials_bound[i]))
        for i in [2, 3, 5, 7, 8, 9]:
            f1 = initial_materials.iloc[:, i] * (1 / (1 + materials_bound[i]))
            f2 = initial_materials.iloc[:, i] * (1 + materials_bound[i])
            materials_lb.iloc[:, i] = np.minimum(f1, f2)
            materials_ub.iloc[:, i] = np.maximum(f1, f2)
        process_lb = initial_process.copy().astype(float)
        process_ub = initial_process.copy().astype(float)
        process_lb.iloc[:, 0] = initial_process.iloc[:, 0] - process_bound[0]
        process_ub.iloc[:, 0] = initial_process.iloc[:, 0] + process_bound[0]

    para_lb = np.concatenate([materials_lb.values.flatten(), process_lb.values.flatten()])
    para_ub = np.concatenate([materials_ub.values.flatten(), process_ub.values.flatten()])
    degenerate = para_lb == para_ub
    para_ub[degenerate] = para_lb[degenerate] + 1e-3
    x_vector_scaled = (x_vector - para_lb) / (para_ub - para_lb)
    file_setting = [n_pure_elements, len(process_bound), len(materials_bound)]
    return {
        'x_vector_scaled': x_vector_scaled,
        'para_lb': para_lb,
        'para_ub': para_ub,
        'process_condition': process_condition,
        'initial_process': initial_process,
        'initial_materials': initial_materials,
        'process_para_name': 'K0',
        'materials_para_name': material_cols,
        'materials_bound': materials_bound,
        'process_bound': process_bound,
        'file_setting': file_setting,
    }


mainfile  = build_config(process_condition, experiment_labels,
                         mainfile_params, dataset_process_defaults)
alloy_ext = AlloyMaterialDependentExtension(mainfile)
params    = setup_parameters(mainfile, mainfile_params, n_pure_elements)

print(f'Unique materials : {alloy_ext.unique}')
print(f'Pure elements    : {alloy_ext.single_el}')
print(f'file_setting     : {params["file_setting"]}')
print(f'Parameter vector : length {len(params["x_vector_scaled"])}')

In [ ]:
# Show how mainfile bounds look for the first pure element (Cr)
mat_cols = params['materials_para_name']
n_mat_params = len(mat_cols)
n_mats = len(alloy_ext.unique)

lb_mainfile = params['para_lb'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)
ub_mainfile = params['para_ub'][:n_mats * n_mat_params].reshape(n_mats, n_mat_params)
init_vals   = params['initial_materials'].values

print('=== Material bounds from mainfile.xlsx (pure elements only) ===')
for i, mat in enumerate(alloy_ext.unique[:n_pure_elements]):
    cmp = pd.DataFrame({
        'param'   : mat_cols,
        'init'    : init_vals[i],
        'lb'      : lb_mainfile[i],
        'ub'      : ub_mainfile[i],
        'btype'   : [mainfile_params['bound_types'].get(p, '?') for p in mat_cols],
        'bmag'    : [mainfile_params['bound_mags'].get(p, '?')  for p in mat_cols],
    })
    print(f'\n  Material: {mat}')
    display(cmp)

## Step 5 — Scale data and prepare tensors

In [ ]:
x_data = fit_data['thickness']
y_data = fit_data['StressThickness']

scaler_x    = MinMaxScaler(feature_range=(0.1, 1.1))
scaler_rawy = MinMaxScaler(feature_range=(0.1, 1.1))
scaler_fity = MinMaxScaler(feature_range=(0.1, 1.1))

x_scaled = scaler_x.fit_transform(x_data.to_numpy().reshape(-1, 1)).flatten()
y_scaled = scaler_rawy.fit_transform(y_data.to_numpy().reshape(-1, 1)).flatten()
scaler_fity.fit(y_data.to_numpy().reshape(-1, 1))

x_tensor         = torch.tensor(x_scaled, dtype=torch.float32).to(DEVICE)
y_tensor         = torch.tensor(y_scaled, dtype=torch.float32).to(DEVICE)
process_tensor   = torch.tensor(params['process_condition'].to_numpy(),
                                dtype=torch.float32).to(DEVICE)
fit_index_tensor = torch.tensor(fit_data['Index'].to_numpy(),
                                dtype=torch.long).to(DEVICE)

print(f'x_tensor shape       : {x_tensor.shape}')
print(f'y_tensor shape       : {y_tensor.shape}')
print(f'process_tensor shape : {process_tensor.shape}')

## Step 6 — Initialize and train the alloy model

In [ ]:
model = AlloySTFModel(
    x0=params['x_vector_scaled'],
    para_lb=params['para_lb'],
    para_ub=params['para_ub'],
    mainfile=mainfile,
    file_setting=params['file_setting'],
).to(DEVICE)

optimizer = torch.optim.LBFGS(
    model.parameters(), lr=0.1, max_iter=20,
    history_size=10, line_search_fn='strong_wolfe'
)
loss_fn = nn.MSELoss()

def closure():
    optimizer.zero_grad()
    y_pred = model(x_tensor, process_tensor, fit_index_tensor, scaler_x, scaler_fity)
    loss = loss_fn(y_pred, y_tensor)
    loss.backward()
    return loss

print('Training alloy model (mainfile bounds)...')
for epoch in range(26):
    loss = optimizer.step(closure)
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d}, Loss: {loss.item() * 100:.5f}')

## Step 7 — Evaluate and plot

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(x_tensor, process_tensor, fit_index_tensor,
                          scaler_x, scaler_fity)

y_pred = scaler_fity.inverse_transform(
    y_pred_scaled.cpu().numpy().reshape(-1, 1)
).flatten()
fit_data = fit_data.copy()
fit_data['y_pred'] = y_pred

print(f'Prediction range  : [{y_pred.min():.3f}, {y_pred.max():.3f}] GPa·nm')
print(f'Experimental range: [{y_data.min():.3f}, {y_data.max():.3f}] GPa·nm')

In [ ]:
melting_vals = params['process_condition']['Melting_T'].unique()
n_plots = len(melting_vals)
n_cols = min(4, n_plots)
n_rows = (n_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
if n_plots == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)

for i, tm in enumerate(melting_vals):
    r, c = divmod(i, n_cols)
    ax = axes[r, c]
    matching_rows = params['process_condition'][
        params['process_condition']['Melting_T'] == tm
    ].index.tolist()
    color_id = 0
    for dataset_id in matching_rows:
        mask = fit_data['Index'] == dataset_id + 1
        thickness = fit_data.loc[mask, 'thickness'].reset_index(drop=True)
        raw_data  = fit_data.loc[mask, 'StressThickness'].reset_index(drop=True)
        pred_data = fit_data.loc[mask, 'y_pred'].reset_index(drop=True)
        ax.plot(thickness, pred_data, color=COLORS[color_id], linewidth=3)
        ax.scatter(thickness, raw_data,  color=COLORS[color_id], s=10)
        color_id = (color_id + 1) % len(COLORS)
    ax.set_title(f'{alloy_ext.unique[i]} (Tm = {tm:.0f} K)')
    ax.set_xlabel('Thickness (nm)')
    ax.set_ylabel('Stress·Thickness (GPa·nm)')
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    ax.tick_params(labelsize=12)

for j in range(n_plots, n_rows * n_cols):
    fig.delaxes(axes.flatten()[j])
plt.suptitle('Alloy extension fitting — mainfile.xlsx bounds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 8 — Display optimized parameters with alloy blending

In [ ]:
with torch.no_grad():
    optimized_scaled = model.x_vector_scaled.cpu().numpy()

para_lb, para_ub = params['para_lb'], params['para_ub']
vector_param = optimized_scaled * (para_ub - para_lb) + para_lb

melting_temps = params['process_condition']['Melting_T'].values
unique_temps  = list(dict.fromkeys(melting_temps))
num_materials = len(unique_temps)

materials_bound = params['materials_bound']
process_bound   = params['process_bound']

mat_count = num_materials * len(materials_bound)
partial_materials = vector_param[:mat_count].reshape(num_materials, len(materials_bound))

# Apply alloy extension: blend A0, B0, l0 for alloy rows from pure elements
n_pure = params['file_setting'][0]
materials_para = alloy_ext.alloy_extension(
    partial_materials, partial_materials[:n_pure, -3:]
)

n_data       = (len(vector_param) - mat_count) // len(process_bound)
process_para = vector_param[mat_count:].reshape(n_data, len(process_bound))

mat_df = pd.DataFrame(materials_para, columns=params['materials_para_name'],
                      index=alloy_ext.unique)
print('Optimized Material Parameters (A0, B0, l0 blended for alloys):')
display(mat_df)

proc_df = pd.DataFrame(process_para, columns=[params['process_para_name']],
                       index=experiment_labels)
print('\nOptimized Process Parameters (K0 per dataset):')
display(proc_df)

## Summary

| Step | Function / Object |
|------|-------------------|
| Inspect raw mainfile | `read_mainfile(path)` → raw DataFrame |
| Parse mainfile       | `parse_mainfile_alloy(path)` → dict |
| Detect pure elements | count rows without `-` in material name |
| Validate order       | pure elements must precede alloys |
| Database loading     | `load_from_database(...)` with `materials` from mainfile |
| File-based loading   | `load_from_mainfile_data(...)` when `filename` column present |
| Alloy extension      | `AlloyMaterialDependentExtension(mainfile)` |
| Apply mainfile bounds| pass `mainfile_params` to `setup_parameters()` |
| Blend alloy params   | `alloy_ext.alloy_extension(partial_materials, pure_energetic)` |

To change initial guesses or bounds without touching Python:
1. Open `alloy_extension_stress_thickness/mainfile.xlsx`
2. Edit values in row 6+ or bound settings in rows 4–5
3. Ensure pure elements remain in the top rows
4. Re-run `fit_alloy_stress.py` or this notebook